In [ ]:
# Problema: Construir documentos de cursos para consultas analíticas que leen curso y resumen de calificaciones juntos.
import json
import sqlite3
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'submission').is_dir())
SOURCE, DATABASE = ROOT / 'data/datacamp_application.sql', ROOT / 'temp/datacamp.db'
OUTPUT = ROOT / 'submission/course_documents.json'
DATABASE.parent.mkdir(exist_ok=True)


In [ ]:
DATABASE.unlink(missing_ok=True)
with sqlite3.connect(DATABASE) as connection:
    connection.executescript(SOURCE.read_text().replace(chr(34) + 'public' + chr(34) + '.', ''))
    courses = pd.read_sql_query("SELECT course_id, title, programming_language FROM courses", connection)
    ratings = pd.read_sql_query("SELECT course_id, COUNT(*) AS rating_count, ROUND(AVG(rating), 2) AS average_rating FROM rating GROUP BY course_id", connection)
courses.shape, ratings.shape


In [ ]:
documents = courses.merge(ratings, on='course_id', how='left').fillna({'rating_count': 0, 'average_rating': 0})
documents['analytics_summary'] = documents.apply(lambda row: {'rating_count': int(row.rating_count), 'average_rating': float(row.average_rating)}, axis=1)
documents = documents.drop(columns=['rating_count', 'average_rating']).to_dict('records')
OUTPUT.write_text(json.dumps(documents, ensure_ascii=False, indent=2), encoding='utf-8')
assert len(documents) == 100
documents[0]
